# 导入必要的库
导入Python标准库和常用数据科学库。

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras import regularizers
import random
import time

d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\s

## 定义类别编码函数
将类别字符串编码为整数，并返回编码后的DataFrame和类别映射字典。

In [2]:
def encode_class_column(df, class_column='ClassName', new_column='num'):
    """
    将类别字符串编码为整数，返回新DataFrame和类别映射字典
    """
    uni = df[class_column].unique()
    mapping = {item: i for i, item in enumerate(uni)}
    df[new_column] = df[class_column].map(mapping)
    return df, mapping

## 评估相关函数
包含GPU设置、数据处理、模型构建、损失函数、训练与多次评估等。

In [ ]:
def set_seed(seed=42):
    tf.keras.backend.clear_session()
    random.seed(seed)
    np.random.seed(seed)
    tf.set_random_seed(seed)

def setup_gpu():
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"检测到{len(gpus)}块GPU，已设置显存按需分配。")
        except RuntimeError as e:
            print(e)
    else:
        print("未检测到GPU，使用CPU运行。")

def split_data(data, test_ratio=0.2):
    idx = np.random.permutation(len(data))
    data_all = data.iloc[idx, :]
    split_idx = int(len(data_all) * (1 - test_ratio))
    train = data_all.iloc[:split_idx]
    test = data_all.iloc[split_idx:]
    return train, test

def get_xy(df):
    # 只保留除 className 和 num 以外的特征列
    feature_cols = [col for col in df.columns if col not in ['className', 'ClassName', 'num']]
    x = df[feature_cols]
    y = df['num'].values
    return x, y

def get_class_map(data):
    return data.drop_duplicates(subset='num').set_index('num')['ClassName'].to_dict()

def build_model(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(512, activation='relu', input_dim=input_dim, kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def multi_category_focal_loss2(gamma=2., alpha=.25, class_weights=None):
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)
    if class_weights is not None:
        weights = np.array([class_weights.get(i, 1.0) for i in range(len(class_weights))])
        class_weights_tf = tf.constant(weights, dtype=tf.float32)
    else:
        class_weights_tf = None
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        y_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.pow(1. - y_t, gamma)
        fl = alpha_t * weight * ce
        if class_weights_tf is not None:
            fl = fl * class_weights_tf
        return tf.reduce_mean(fl)
    return focal_loss_fixed

def train_and_evaluate(train_df, test_df, class_map, verbose=0):
    x_train, y_train = get_xy(train_df)
    x_test, y_test = get_xy(test_df)
    num_classes = len(class_map)
    # 转成one-hot编码
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
    class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
    class_weights = dict(enumerate(class_weights_array))
    model = build_model(x_train.shape[1], num_classes)
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    model.compile(loss=multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights),
                  optimizer=optimizer, metrics=['accuracy'])
    early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
    model.fit(x_train, y_train_cat, epochs=500, batch_size=64, verbose=verbose, callbacks=[reduce_lr, early_stopping])
    y_pred = model.predict(x_test).argmax(axis=1)
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_test, y_pred, labels=list(class_map.keys()))
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    return [acc, precision, recall, f1], per_class_acc

def multi_run_eval(data, mapping_df, n_runs=10, test_ratio=0.2, seed=42):
    set_seed(seed)
    class_map = get_class_map(data)
    all_class_ids = sorted(class_map.keys())
    class_labels = [class_map[i] for i in all_class_ids]
    num2veg = dict(zip(mapping_df['num'], mapping_df['Veg_Formation']))
    overall_scores = []
    per_class_accs = []
    overall_veg_scores = []
    for run in range(n_runs):
        print(f"\n===== 正在进行第 {run+1}/{n_runs} 次训练与评估 =====")
        train_df, test_df = split_data(data, test_ratio)
        x_train, y_train = get_xy(train_df)
        x_test, y_test = get_xy(test_df)
        num_classes = len(class_map)
        y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
        class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
        class_weights = dict(enumerate(class_weights_array))
        model = build_model(x_train.shape[1], num_classes)
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
        model.compile(loss=multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights),
                      optimizer=optimizer, metrics=['accuracy'])
        early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
        model.fit(x_train, y_train_cat, epochs=500, batch_size=64, verbose=0, callbacks=[reduce_lr, early_stopping])
        y_pred_num = model.predict(x_test).argmax(axis=1)
        # 细类四项指标
        acc = accuracy_score(y_test, y_pred_num)
        precision = precision_score(y_test, y_pred_num, average='macro', zero_division=0)
        recall = recall_score(y_test, y_pred_num, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred_num, average='macro', zero_division=0)
        cm = confusion_matrix(y_test, y_pred_num, labels=list(class_map.keys()))
        per_class_acc = cm.diagonal() / cm.sum(axis=1)
        # 大类指标
        y_true_veg = [num2veg[n] for n in y_test]
        y_pred_veg = [num2veg[n] for n in y_pred_num]
        acc_veg = accuracy_score(y_true_veg, y_pred_veg)
        precision_veg = precision_score(y_true_veg, y_pred_veg, average='macro', zero_division=0)
        recall_veg = recall_score(y_true_veg, y_pred_veg, average='macro', zero_division=0)
        f1_veg = f1_score(y_true_veg, y_pred_veg, average='macro', zero_division=0)
        overall_veg_scores.append([acc_veg, precision_veg, recall_veg, f1_veg])
        if len(per_class_acc) < len(all_class_ids):
            acc_full = np.full(len(all_class_ids), np.nan)
            acc_full[:len(per_class_acc)] = per_class_acc
            per_class_accs.append(acc_full)
        else:
            per_class_accs.append(per_class_acc)
        overall_scores.append([acc, precision, recall, f1])
        print(f"Run {run+1}: acc={acc:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")
        print(f"Veg_Formation: acc={acc_veg:.4f}, precision={precision_veg:.4f}, recall={recall_veg:.4f}, f1={f1_veg:.4f}")
    # 汇总表格
    score_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    veg_score_names = ['Veg_Formation_Accuracy', 'Veg_Formation_Precision', 'Veg_Formation_Recall', 'Veg_Formation_F1']
    index = score_names + veg_score_names + class_labels
    results = np.vstack([
        np.array(overall_scores).T,
        np.array(overall_veg_scores).T,
        np.array(per_class_accs).T
    ])
    df_results = pd.DataFrame(results, index=index, columns=[f'Run_{i+1}' for i in range(n_runs)])
    df_results['Mean'] = df_results.mean(axis=1)
    print('\n多次整体与各类别准确率统计表：')
    print(df_results)
    return df_results

## 定义主流程函数
包含数据加载、类别编码、模型训练与评估等完整流程。

In [ ]:
def main(code_csv_path, mapping_csv_path, result_csv_path, n_runs=10, test_ratio=0.2, seed=42):
    """
    读取原始特征文件和类别映射表，将类别映射为num后评估，并统计大类指标
    """
    # 读取原始特征数据
    df = pd.read_csv(code_csv_path, encoding='utf_8_sig')
    # 读取类别映射表
    mapping_df = pd.read_csv(mapping_csv_path, encoding='utf_8_sig')
    mapping = dict(zip(mapping_df['ClassName'], mapping_df['num']))
    # 添加num列
    df['num'] = df['ClassName'].map(mapping)
    # 检查是否有未映射的类别
    if df['num'].isnull().any():
        raise ValueError('有ClassName未能映射到num，请检查类别映射表！')
    # 多次评估，传入mapping_df用于大类统计
    df_results = multi_run_eval(df, mapping_df, n_runs=n_runs, test_ratio=test_ratio, seed=seed)
    # 保存评估结果
    df_results.to_csv(result_csv_path, encoding='utf_8_sig')
    # 生成数字到类别的反向映射
    num2class = dict(zip(mapping_df['num'], mapping_df['ClassName']))
    return num2class, df_results

## 调用主流程函数并输出结果
请根据实际数据路径修改参数。

In [ ]:
# 示例：请根据实际路径修改
code_csv = 'F:/TensorFlow/xinjiang/traindata20250626_2.csv'
mapping_csv = 'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv'
result_csv = 'F:/TensorFlow/xinjiang/XJruns20250627_50.csv'

num2class, df_results = main(code_csv, mapping_csv, result_csv, n_runs=50, test_ratio=0.2, seed=42)
print('数字到类别映射:', num2class)
display(df_results)



===== 正在进行第 1/50 次训练与评估 =====
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor

===== 正在进行第 1/50 次训练与评估 =====
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Run 1: acc=0.4194, precision=0.3272, recall=0.3852, f1=0.3348
Veg_Formation: acc=0.5914, precision=0.4913, recall=0.4524, f1=0.4655

===== 正在进行第 2/50 次训练与评估 =====
Run 1: acc=0.4194, precision=0.3272, recall=0.3852, f1=0.3348
Veg_Formation: acc=0.5914, precision=0.4913, recall=0.4524, f1=0.4655

===== 正在进行第 2/50 次训练与评估 =====


d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:130: RuntimeWarning: invalid value encountered in true_divide


Run 2: acc=0.4301, precision=0.3140, recall=0.3893, f1=0.3004
Veg_Formation: acc=0.5806, precision=0.4093, recall=0.4063, f1=0.4049

===== 正在进行第 3/50 次训练与评估 =====


d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:130: RuntimeWarning: invalid value encountered in true_divide


Run 3: acc=0.4552, precision=0.3101, recall=0.3173, f1=0.2851
Veg_Formation: acc=0.6344, precision=0.4773, recall=0.4507, f1=0.4553

===== 正在进行第 4/50 次训练与评估 =====
Run 4: acc=0.4337, precision=0.2852, recall=0.3122, f1=0.2856
Veg_Formation: acc=0.6237, precision=0.5185, recall=0.5299, f1=0.5207

===== 正在进行第 5/50 次训练与评估 =====
Run 4: acc=0.4337, precision=0.2852, recall=0.3122, f1=0.2856
Veg_Formation: acc=0.6237, precision=0.5185, recall=0.5299, f1=0.5207

===== 正在进行第 5/50 次训练与评估 =====
Run 5: acc=0.4588, precision=0.2917, recall=0.3647, f1=0.3063
Veg_Formation: acc=0.6022, precision=0.4542, recall=0.4764, f1=0.4564

===== 正在进行第 6/50 次训练与评估 =====
Run 5: acc=0.4588, precision=0.2917, recall=0.3647, f1=0.3063
Veg_Formation: acc=0.6022, precision=0.4542, recall=0.4764, f1=0.4564

===== 正在进行第 6/50 次训练与评估 =====
Run 6: acc=0.4659, precision=0.3220, recall=0.3646, f1=0.3214
Veg_Formation: acc=0.6344, precision=0.5529, recall=0.4890, f1=0.5015

===== 正在进行第 7/50 次训练与评估 =====
Run 6: acc=0.4659, pre

...

d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:130: RuntimeWarning: invalid value encountered in true_divide


Run 44: acc=0.4480, precision=0.3070, recall=0.3960, f1=0.3210
Veg_Formation: acc=0.6093, precision=0.4907, recall=0.5776, f1=0.5043

===== 正在进行第 45/50 次训练与评估 =====
Run 45: acc=0.4229, precision=0.2655, recall=0.3459, f1=0.2809
Veg_Formation: acc=0.6022, precision=0.5285, recall=0.5315, f1=0.5227

===== 正在进行第 46/50 次训练与评估 =====
Run 45: acc=0.4229, precision=0.2655, recall=0.3459, f1=0.2809
Veg_Formation: acc=0.6022, precision=0.5285, recall=0.5315, f1=0.5227

===== 正在进行第 46/50 次训练与评估 =====
Run 46: acc=0.4731, precision=0.3148, recall=0.3277, f1=0.2745
Veg_Formation: acc=0.6487, precision=0.4689, recall=0.4634, f1=0.4641

===== 正在进行第 47/50 次训练与评估 =====
Run 46: acc=0.4731, precision=0.3148, recall=0.3277, f1=0.2745
Veg_Formation: acc=0.6487, precision=0.4689, recall=0.4634, f1=0.4641

===== 正在进行第 47/50 次训练与评估 =====
Run 47: acc=0.4086, precision=0.3200, recall=0.3182, f1=0.2956
Veg_Formation: acc=0.5591, precision=0.4622, recall=0.4432, f1=0.4441

===== 正在进行第 48/50 次训练与评估 =====
Run 47: ac

d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\ipykernel_launcher.py:130: RuntimeWarning: invalid value encountered in true_divide


Run 48: acc=0.4373, precision=0.2954, recall=0.3052, f1=0.2689
Veg_Formation: acc=0.6165, precision=0.5117, recall=0.5415, f1=0.5183

===== 正在进行第 49/50 次训练与评估 =====
Run 49: acc=0.4444, precision=0.3362, recall=0.4224, f1=0.3506
Veg_Formation: acc=0.6201, precision=0.5065, recall=0.5288, f1=0.5117

===== 正在进行第 50/50 次训练与评估 =====
Run 49: acc=0.4444, precision=0.3362, recall=0.4224, f1=0.3506
Veg_Formation: acc=0.6201, precision=0.5065, recall=0.5288, f1=0.5117

===== 正在进行第 50/50 次训练与评估 =====
Run 50: acc=0.4588, precision=0.3728, recall=0.3963, f1=0.3602
Veg_Formation: acc=0.6344, precision=0.5367, recall=0.5651, f1=0.5383

多次整体与各类别准确率统计表：
                            Run_1     Run_2     Run_3     Run_4     Run_5  \
Accuracy                 0.419355  0.430108  0.455197  0.433692  0.458781   
Precision                0.327240  0.313970  0.310134  0.285154  0.291693   
Recall                   0.385229  0.389314  0.317280  0.312185  0.364681   
F1 Score                 0.334812  0.300409  0.

,Run_1,Run_2,Run_3,Run_4,Run_5,Run_6,Run_7,Run_8,Run_9,Run_10,...,Run_42,Run_43,Run_44,Run_45,Run_46,Run_47,Run_48,Run_49,Run_50,Mean
Accuracy,0.419355,0.430108,0.455197,0.433692,0.458781,0.465950,0.473118,0.444444,0.412186,0.448029,...,0.465950,0.455197,0.448029,0.422939,0.473118,0.408602,0.437276,0.444444,0.458781,0.439713
Precision,0.327240,0.313970,0.310134,0.285154,0.291693,0.322040,0.359550,0.335761,0.276186,0.293233,...,0.322126,0.328600,0.306977,0.265464,0.314751,0.319989,0.295428,0.336174,0.372847,0.317037
Recall,0.385229,0.389314,0.317280,0.312185,0.364681,0.364610,0.369929,0.451011,0.321617,0.324345,...,0.351774,0.325767,0.395955,0.345886,0.327680,0.318181,0.305232,0.422448,0.396252,0.352121
F1 Score,0.334812,0.300409,0.285148,0.285599,0.306282,0.321384,0.340108,0.334165,0.274110,0.290826,...,0.324451,0.294585,0.321043,0.280917,0.274492,0.295574,0.268851,0.350559,0.360182,0.304827
Veg_Formation_Accuracy,0.591398,0.580645,0.634409,0.623656,0.602151,0.634409,0.623656,0.616487,0.577061,0.612903,...,0.620072,0.605735,0.609319,0.602151,0.648746,0.559140,0.616487,0.620072,0.634409,0.605448
Veg_Formation_Precision,0.491326,0.409286,0.477284,0.518526,0.454211,0.552861,0.552705,0.484862,0.436997,0.451859,...,0.463198,0.583451,0.490687,0.528550,0.468898,0.462177,0.511676,0.506518,0.536706,0.488742
Veg_Formation_Recall,0.452430,0.406283,0.450679,0.529854,0.476366,0.488965,0.570786,0.573190,0.528696,0.470793,...,0.519031,0.468183,0.577624,0.531466,0.463421,0.443164,0.541520,0.528825,0.565081,0.510671
Veg_Formation_F1,0.465512,0.404904,0.455341,0.520673,0.456402,0.501542,0.554749,0.476661,0.435087,0.448301,...,0.472978,0.490646,0.504337,0.522731,0.464079,0.444109,0.518264,0.511685,0.538261,0.483305
针茅属荒漠草原,0.052632,0.117647,0.000000,0.090909,0.076923,0.181818,0.000000,0.214286,0.200000,0.090909,...,0.062500,0.000000,0.176471,0.066667,0.052632,0.428571,0.166667,0.142857,0.157895,0.130648
紫花针茅草原,0.666667,0.333333,0.600000,0.500000,0.550000,0.666667,0.000000,0.600000,0.357143,0.555556,...,0.600000,0.555556,0.666667,0.444444,0.888889,0.700000,0.470588,0.500000,0.222222,0.510395
